Now we reach the core Computer Vision concept: Convolutional Neural Networks (CNNs).

You've already learned a normal neural network:

``` text

Input
 ↓
Linear
 ↓
ReLU
 ↓
Linear
 ↓
Output

```
A CNN uses the same neural-network principles—weights, bias, activations, loss, backpropagation, optimizer—but introduces special layers designed to understand the spatial structure of images.

Our goal in this notebook is to understand:

``` text

Image
 ↓
Convolution
 ↓
Feature Map
 ↓
ReLU
 ↓
Pooling
 ↓
Repeat
 ↓
Learned visual features
 ↓
Classifier
 ↓
Defect class

```

##### 1. Why not just use our previous neural network?

Suppose our image is: RGB image = [3, 224, 224]

That's: 3 × 224 × 224 = 150,528 values

We could flatten it: [3,224,224] -->  flatten [150,528]

and feed it into: nn.Linear(150528, 128)

But there's a problem.

An image isn't just 150,528 unrelated numbers.

The positions and neighboring relationships of pixels matter.

For example:

■ ■ ■ ■ ■

could form a line.

The same pixels scattered around:

■       ■

    ■

       ■   ■

represent a completely different visual pattern.

That's the spatial structure we discussed earlier.

CNNs are specifically designed to preserve and learn from these local spatial relationships.

##### 2. What does "convolution" mean?

Let's use a tiny grayscale image so we can calculate it manually.

Suppose we have:

Image

``` text

1  1  1  0  0
1  1  1  0  0
1  1  1  0  0
0  0  0  1  1
0  0  0  1  1

This is a: 5 × 5 image

Now introduce a tiny matrix:

1  0 -1
1  0 -1
1  0 -1

```

This is called a: kernel or filter

Its size is: 3 × 3

We'll slide this small filter across the image.

##### 3. What does the filter actually do?

Take the first 3 × 3 region of our image:

``` text

1  1  1
1  1  1
1  1  1

```

Our filter is:
``` text

 1   0  -1
 1   0  -1
 1   0  -1

```
Multiply corresponding positions:

``` text

Image          Filter

1 1 1          1  0 -1
1 1 1    ×     1  0 -1
1 1 1          1  0 -1

```

Calculate:

``` text

(1×1) + (1×0) + (1×-1)
+
(1×1) + (1×0) + (1×-1)
+
(1×1) + (1×0) + (1×-1)

```

which becomes:

1 + 0 - 1
+
1 + 0 - 1
+
1 + 0 - 1

= 0

So the output for this location is: 0

##### 4. Then the filter moves

We move the filter one position to the right.

Now it covers:

``` text

1  1  0
1  1  0
1  1  0

Apply the same filter:

Image          Filter

1 1 0          1  0 -1
1 1 0    ×     1  0 -1
1 1 0          1  0 -1

Calculate:

(1×1) + (1×0) + (0×-1)
+
(1×1) + (1×0) + (0×-1)
+
(1×1) + (1×0) + (0×-1)

= 3

````
Now we get: 3

That's a stronger response.

Why?

Because this region contains a strong change:

``` text

1 1 | 0
1 1 | 0
1 1 | 0

```

There is effectively an edge there.

So this particular filter responds strongly to a certain kind of edge.

##### 5. This is the central CNN idea

A filter looks at a small local region of an image.

Whole image

``` text

┌─────────────────────┐
│                     │
│     ┌───────┐       │
│     │ 3 × 3 │       │
│     │region │       │
│     └───────┘       │
│                     │
└─────────────────────┘

It performs:


local pixels
    ↓
multiply by filter weights
    ↓
sum
    ↓
one output value

Then:

move filter
    ↓
calculate again
    ↓
move filter
    ↓
calculate again

```

This process is convolution.

##### 6. What is a feature map?

As the filter moves across the entire image, we collect all the output values.

For example:

``` text

0   3   3
0   3   3
...

That output matrix is called a: Feature map

So:

Image
  ↓
Filter slides over image
  ↓
Convolution
  ↓
Feature map

```

The feature map tells us: Where did this filter detect the pattern it responds to?

If a filter responds strongly to vertical edges, high values in its feature map indicate locations containing those edge-like patterns.

##### 7. Very important: do we manually create these filters?

This is where CNNs become much more interesting.

Below is a simple illustrative edge filter.

``` text

1  0 -1
1  0 -1
1  0 -1

```

But in a real CNN, we usually do not manually define the filter values.

Instead, the filter values are: learnable weights

This connects directly to your neural-network learning.

Initially:

Filter

``` text

?  ?  ?
?  ?  ?
?  ?  ?

```
The values start from an initialization.

During training:

``` text

Image
 ↓
CNN
 ↓
Prediction
 ↓
Loss
 ↓
Backpropagation
 ↓
Gradients
 ↓
Optimizer
 ↓
Update filter weights

```

After many examples, useful filters emerge.

So:

CNN filters are learned through backpropagation just like the weights in the neural network.

This is a major connection.

##### 8. A CNN doesn't have only one filter

Suppose the first convolutional layer has: 32 filters

Each filter can learn a different pattern.

Conceptually:

``` text

Image
  │
  ├── Filter 1 → Feature Map 1
  │
  ├── Filter 2 → Feature Map 2
  │
  ├── Filter 3 → Feature Map 3
  │
  ...
  └── Filter 32 → Feature Map 32

```
Therefore the layer produces: 32 feature maps

Different filters may become sensitive to different visual patterns.

For example, simplified:

- Filter A → vertical-like edges
- Filter B → horizontal-like edges
- Filter C → diagonal-like patterns
- Filter D → texture-like patterns
- ...

We don't normally tell each filter what to learn.

Training determines the useful weights.

##### 9. This is similar to neurons

There's a useful connection to neural network.

In a regular layer: nn.Linear(384, 64)

you learned that there are: 64 neurons

Each neuron has learnable weights.

In a convolution layer:

nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3
)

we're asking for: 32 output channels which means the layer learns 32 filters.

So conceptually:
``` text

Linear layer
64 neurons
→ learn different combinations of input features

CNN layer
32 filters
→ learn different local visual patterns

```

Not identical mechanisms, but the learning principle is very similar.

##### 10. What does kernel_size=3 mean?

When you see:

nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3
)

kernel_size=3 means: 3 × 3 spatial filter.

But our RGB image has: 3 channels

So an actual first-layer filter must look through all three input channels.

Its weight shape is effectively: 

[3, 3, 3]

 ↑  ↑  ↑
 C  H  W

You can picture one filter as:
``` text

Red part       Green part       Blue part

3 × 3           3 × 3           3 × 3

```
Together, those channel-specific weights form one filter acting across the RGB input.

##### 11. Shape example

Suppose our input batch is: [32, 3, 224, 224]

32  = images
3   = RGB channels
224 = height
224 = width

Then:

conv = nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3
)

means:

Input has: 3 channels
Layer learns: 32 filters

Each filter: 3 input channels × 3 × 3 spatial kernel

Ignoring bias for a moment, each filter contains: 3 × 3 × 3 = 27 weights

Since we have 32 filters: 27 × 32 = 864 weights

If each filter also has one bias: 864 + 32 = 896 trainable parameters

Those parameters are learned by training.

##### 12. What is stride?

So far we've moved the filter: 1 pixel at a time

That's: stride = 1

Example:

``` text

Position 1
[XXX].....

Position 2
.[XXX]....

Position 3
..[XXX]...

```
With:

stride = 2

we jump two positions:

``` text

Position 1
[XXX].....

Position 2
..[XXX]...

Position 3
....[XXX]

```
So:

**Stride = how many pixels the filter moves at each step.**

Larger stride generally reduces the spatial dimensions of the resulting feature map.

##### 13. What is padding?

Suppose our image is: 5 × 5

and filter: 3 × 3

Without padding, the filter can't be centered on the border pixels while staying entirely inside the image.

One solution is to add a border around the image:

``` text

0 0 0 0 0 0 0
0 1 1 1 0 0 0
0 1 1 1 0 0 0
0 1 1 1 0 0 0
0 0 0 0 1 1 0
0 0 0 0 1 1 0
0 0 0 0 0 0 0

```
Those added values are padding.

For a common 3×3 convolution:

nn.Conv2d(
    in_channels=3,
    out_channels=32,
    kernel_size=3,
    padding=1
)

padding of 1 adds one pixel around the border.

With:

kernel = 3
stride = 1
padding = 1

we can commonly preserve spatial size:

224 × 224
     ↓
Convolution
     ↓
224 × 224

##### 14. Then comes ReLU

ReLU(x) = max(0, x)

For example: Before ReLU

``` text

-3   2   5
 1  -4   7

      ↓ ReLU

 0   2   5
 1   0   7

```

So CNNs haven't introduced a new ReLU concept.

It's the same nonlinear activation already used.

A common pattern is:

``` text

Convolution
     ↓
Feature maps
     ↓
ReLU

```

##### 15. Pooling

Now suppose a feature map contains:

``` text 

1  3  2  0
4  6  1  2
2  1  5  3
0  2  4  8

Let's use: MaxPool 2 × 2

Take the first region:

1  3
4  6

Maximum: 6

Next:

2  0
1  2

Maximum: 2

Bottom-left:

2  1
0  2

Maximum: 2

Bottom-right:

5  3
4  8

Maximum: 8

Result:

6  2
2  8

We transformed:

4 × 4
 ↓
2 × 2

```
That's max pooling.

##### 16. Why pool?

Pooling reduces spatial dimensions while retaining strong local responses.

Conceptually:

``` text

Large feature map
       ↓
Max pooling
       ↓
Smaller feature map

```

Benefits include reducing:

- computation
- memory
- spatial dimensions

while retaining prominent detected features.

For now, think:

**Convolution detects local patterns; pooling summarizes/downsamples local responses.**

##### 17. Put one CNN block together

A common CNN block is:

``` text

Image / previous feature maps
             ↓
         Conv2d
             ↓
          ReLU
             ↓
         MaxPool
             ↓
smaller learned feature maps

```

Then we can repeat:

``` text

Input image
    ↓
Conv
    ↓
ReLU
    ↓
Pool
    ↓
Conv
    ↓
ReLU
    ↓
Pool
    ↓
...

```

##### 18. How does this eventually recognize a defect?

This is one of the most important conceptual pieces.

Early CNN layers work with very local information.

Simplified:

``` text

Pixels
  ↓
early convolution layers
  ↓
edges / simple patterns
  ↓
middle layers
  ↓
textures / combinations of patterns
  ↓
deeper layers
  ↓
more complex visual structures
  ↓
classifier
  ↓
Scratch / Crack / Dent / ...

The network builds increasingly useful representations.

For a scratch, conceptually:

pixel differences
      ↓
edge-like patterns
      ↓
line-like structures
      ↓
surface texture + line combinations
      ↓
scratch-related representation
      ↓
Scratch

```

Important nuance: we don't usually assign these meanings manually to particular filters. They're learned representations, and real networks can be more complicated than this simplified hierarchy.

##### 19. How does classification happen at the end?

Eventually the CNN produces learned visual features.

Those features can be fed into a classifier.

For example:

``` text

CNN feature extraction
       ↓
Learned features
       ↓
Linear layer
       ↓
4 logits

```
Suppose we have four classes:

- 0 = Crack
- 1 = Dent
- 2 = Scratch
- 3 = Normal

Output:

logits --> [-1.2, 0.4, 3.1, -0.7]

- Largest logit: 3.1
- index: 2
- prediction: Scratch

This is exactly the multiclass-classification logic in NLP neural network.

##### 20. Loss function

For multiclass image classification, we'll likely use: nn.CrossEntropyLoss()

Same idea as in four-class NLP neural network.

``` text

CNN
 ↓
logits
 ↓
CrossEntropyLoss(logits, true_label)
 ↓
loss
 ↓
backpropagation

```

And remember:

CrossEntropyLoss expects raw logits, so we don't need to manually apply softmax before passing outputs to the loss.

##### 21. How do convolution filters actually learn?

Initially: Filter weights ≈ initialized values

First training batch:

``` text

Images
 ↓
Convolution
 ↓
Feature maps
 ↓
CNN
 ↓
Logits
 ↓
Loss

Then:

loss.backward()

computes gradients.

Those gradients reach the convolutional filters too.

Then:

optimizer.step()

updates their weights.

So:

CNN filter
   ↓
is made of weights
   ↓
weights receive gradients
   ↓
optimizer updates them
   ↓
filter gradually becomes useful

```
You already know the underlying learning mechanism.

**CNN does not replace backpropagation. CNN adds an image-oriented architecture whose parameters are learned using backpropagation.**

##### 22. Connection to NLP neural network

Your NLP model:

``` text

Ticket
 ↓
MiniLM embedding
 ↓
384 numbers
 ↓
Linear 384 → 64
 ↓
ReLU
 ↓
Linear 64 → 32
 ↓
ReLU
 ↓
Linear 32 → 4
 ↓
logits

```

Our image CNN:

``` text

Image
 ↓
[3,224,224]
 ↓
Conv
 ↓
ReLU
 ↓
Pool
 ↓
Conv
 ↓
ReLU
 ↓
Pool
 ↓
learned visual features
 ↓
Linear
 ↓
class logits

```

Training for both:

``` text

forward
 ↓
loss
 ↓
backprop
 ↓
optimizer
 ↓
update weights

```

So you're not learning deep learning again.

You're learning a new architecture for a new kind of input.

##### 23. 🔍 XAI connection

CNNs introduce an interesting question.

Suppose:

- Prediction: Scratch
- Confidence: 97%

We know internally the CNN generated many feature maps.

But which **part of the original image** most influenced the Scratch prediction?

Later Grad-CAM will help us visualize something conceptually like:

Product image

``` text

┌─────────────────────────┐
│                         │
│       █████████         │
│       █ scratch █       │
│       █ region  █       │
│       █████████         │
│                         │
└─────────────────────────┘

```

That's:

🔍 Explainable AI

What image regions strongly influenced the prediction?

Understanding convolution and feature maps first will make Grad-CAM much less mysterious later.

##### 24. ⚖️ RAI connection

Suppose our CNN predicts scratches correctly because every scratch-training image contains a particular company sticker in one corner.

The model could learn:

``` text

Sticker
   ↓
Scratch

```
instead of:

``` text

Actual surface defect
   ↓
Scratch

```
Training accuracy could still look excellent.

This illustrates why:

High accuracy ≠ proof that the model learned the right thing

Later:

``` text

🔍 XAI
     ↓
Where is model looking?
     ↓
Sticker!
     ↓
⚖️ RAI
     ↓
Potential reliability problem

```

##### 25. The most important CNN vocabulary

| Term                | Simple meaning                                                         |
| ------------------- | ---------------------------------------------------------------------- |
| **Kernel / Filter** | Small set of learned weights that scans local image regions            |
| **Convolution**     | Applying the filter across the image                                   |
| **Feature map**     | Output showing responses of a filter across spatial locations          |
| **Stride**          | How far the filter moves each step                                     |
| **Padding**         | Extra border values added around an image/feature map                  |
| **ReLU**            | Nonlinear activation that replaces negative values with zero           |
| **Pooling**         | Downsamples/summarizes local feature responses                         |
| **Channel**         | One component/feature plane of an image or intermediate representation |
| **CNN**             | Neural network architecture designed to learn spatial visual patterns  |


##### 26. One diagram to remember


``` text

RGB IMAGE
[3,224,224]
     │
     ▼
┌─────────────┐
│ Convolution │ ← learned filters scan local regions
└─────────────┘
     │
     ▼
 FEATURE MAPS
     │
     ▼
    ReLU
     │
     ▼
   Pooling
     │
     ▼
 smaller feature maps
     │
     ▼
 More CNN layers
     │
     ▼
 increasingly useful
 visual representations
     │
     ▼
  Classifier
     │
     ▼
    LOGITS
     │
     ▼
 Crack / Dent /
 Scratch / Normal

 ```

##### Key Learnings

1. CNN stands for Convolutional Neural Network.

2. CNNs are designed to learn spatial patterns from images.

3. A kernel/filter is a small collection of weights that examines
   local regions of an image.

4. Convolution applies a filter across different spatial locations.

5. The output produced by a filter across the image is a feature map.

6. CNN filters are normally learned during training rather than
   manually defined.

7. Multiple filters allow a convolutional layer to learn multiple
   types of visual patterns.

8. Stride controls how far a filter moves at each step.

9. Padding adds border values and can help control feature-map size.

10. ReLU introduces nonlinearity just as it did in our previous
    neural networks.

11. Pooling reduces spatial dimensions while retaining important
    local responses.

12. CNN layers can build increasingly useful representations from
    simple local patterns toward more complex visual patterns.

13. CNN parameters are learned through the same:
       loss → backpropagation → optimizer
    process already learned in Deep Learning and NLP.

14. Multiclass image classification can produce logits and use
    CrossEntropyLoss, just like our multiclass NLP classifier.

15. 🔍 XAI helps investigate which image regions contributed to
    a CNN prediction.

16. ⚖️ RAI asks whether the model learned meaningful defect
    information rather than unreliable shortcuts.

**A CNN learns small filters that scan neighboring pixels, producing feature maps that represent where useful visual patterns occur. Those filter weights are learned through backpropagation.**